In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 6 - WEEK 9 BAYESIAN OPTIMISATION
# Run from inside the week9/ folder
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 8 calibration check
# ------------------------------------------------------------

week8_pred_mean = -0.192818
week8_pred_std = 0.020270
week8_actual = -0.27148587196676827

week8_error = week8_actual - week8_pred_mean
week8_z_error = week8_error / week8_pred_std

print("\n================================")
print("WEEK 8 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week8_pred_mean)
print("Predicted std :", week8_pred_std)
print("Actual        :", week8_actual)

print("\nPrediction error:")
print(week8_error)

print("\nError / predicted std:")
print(week8_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(5) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Global/local candidate search
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

local_candidates = (
    best_x
    + rng.normal(0, local_scale, size=(100000, 5))
)

wide_candidates = (
    best_x
    + rng.normal(0, wide_scale, size=(70000, 5))
)

global_candidates = rng.uniform(
    0,
    1,
    size=(150000, 5)
)

local_candidates = np.clip(local_candidates, 0, 1)
wide_candidates = np.clip(wide_candidates, 0, 1)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

mu, sigma = gp.predict(
    candidates,
    return_std=True
)

print("\nGlobal/local candidates:", len(candidates))


# ------------------------------------------------------------
# 6. Global diagnostics
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)
mean_idx = np.argmax(mu)

print("\n================================")
print("GLOBAL DIAGNOSTICS")
print("================================")

print("\nPrimary EI:")
print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])

print("\nGlobal UCB:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 7. Conservative ARD trust region
# ------------------------------------------------------------
#
# Week 8 was badly miscalibrated, so we also search close
# to the best ACTUALLY OBSERVED point.
#
# Widths come automatically from the fitted ARD lengthscales.
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.25 * lengthscales,
    0.025,
    0.10
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("TRUST REGION")
print("================================")

print("Best observed:", best_x)
print("Half-widths:", trust_half_width)
print("Lower:", lower)
print("Upper:", upper)


# ------------------------------------------------------------
# 8. Trust-region candidate pool
# ------------------------------------------------------------

tr_candidates = rng.uniform(
    lower,
    upper,
    size=(200000, 5)
)

distance, _ = tree.query(
    tr_candidates,
    k=1
)

tr_candidates = tr_candidates[
    distance > 0.01
]

tr_mu, tr_sigma = gp.predict(
    tr_candidates,
    return_std=True
)

tr_EI = expected_improvement(
    tr_mu,
    tr_sigma,
    best_y,
    xi=0.0
)

tr_ei_idx = np.argmax(tr_EI)
tr_mean_idx = np.argmax(tr_mu)

print("\n================================")
print("TRUST-REGION RESULTS")
print("================================")

print("\nTrust-region EI:")
print("candidate =", tr_candidates[tr_ei_idx])
print("mean =", tr_mu[tr_ei_idx])
print("std =", tr_sigma[tr_ei_idx])
print("EI =", tr_EI[tr_ei_idx])

print("\nTrust-region highest mean:")
print("candidate =", tr_candidates[tr_mean_idx])
print("mean =", tr_mu[tr_mean_idx])
print("std =", tr_sigma[tr_mean_idx])

print("\nTrust-region UCB:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = tr_mu + beta * tr_sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", tr_candidates[idx],
        "\n mean =", round(tr_mu[idx], 6),
        "\n std =", round(tr_sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )

X shape: (28, 5)
Y shape: (28,)

Current best:
[0.442929 0.409333 0.63183  0.7398   0.129381] -> -0.19517557780237

Y range:
min = -2.5711696316081234
max = -0.19517557780237
std = 0.640158855323448

WEEK 8 CALIBRATION CHECK
Predicted mean: -0.192818
Predicted std : 0.02027
Actual        : -0.27148587196676827

Prediction error:
-0.07866787196676828

Error / predicted std:
-3.8810000970285286

GP FIT

Fitted kernel:
1.28**2 * Matern(length_scale=[0.815, 0.974, 1.27, 0.899, 1.03], nu=2.5) + WhiteKernel(noise_level=0.00268)

ARD lengthscales:
[0.81509305 0.97382103 1.27095006 0.89856583 1.02729773]

Normalised inverse-lengthscale sensitivity:
[0.23929918 0.20029461 0.15346874 0.21706935 0.18986813]

Global/local candidates: 319997

GLOBAL DIAGNOSTICS

Primary EI:
candidate = [0.4361678  0.35602656 0.58005235 0.66569633 0.12327135]
mean = -0.21477268063614496
std = 0.054596487931781554
EI = 0.013370556630139927

Highest predicted mean:
candidate = [0.43981406 0.36954167 0.59739181 0.68565

In [2]:
# ============================================================
# FUNCTION 6 WEEK 9 - TIGHTER CALIBRATION-AWARE TRUST REGION
# ============================================================
#
# Week 8 missed by -3.88 predictive standard deviations.
# We therefore contract the search around the best ACTUALLY
# observed point before selecting the Week 9 query.
#
# Widths still come from ARD lengthscales, but are capped at
# 0.05 rather than 0.10 because recent local calibration was poor.

tight_half_width = np.clip(
    0.125 * lengthscales,
    0.025,
    0.05
)

tight_lower = np.maximum(
    0.0,
    best_x - tight_half_width
)

tight_upper = np.minimum(
    1.0,
    best_x + tight_half_width
)

print("Best observed:", best_x)
print("Tight half-widths:", tight_half_width)
print("Lower:", tight_lower)
print("Upper:", tight_upper)


# ------------------------------------------------------------
# Candidate search
# ------------------------------------------------------------

tight_candidates = rng.uniform(
    tight_lower,
    tight_upper,
    size=(250000, 5)
)

distance, _ = tree.query(
    tight_candidates,
    k=1
)

tight_candidates = tight_candidates[
    distance > 0.01
]

tight_mu, tight_sigma = gp.predict(
    tight_candidates,
    return_std=True
)


# ------------------------------------------------------------
# EI
# ------------------------------------------------------------

tight_EI = expected_improvement(
    tight_mu,
    tight_sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(tight_EI)

print("\nTIGHT EI:")
print("candidate =", tight_candidates[ei_idx])
print("mean =", tight_mu[ei_idx])
print("std =", tight_sigma[ei_idx])
print("EI =", tight_EI[ei_idx])


# ------------------------------------------------------------
# Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(tight_mu)

print("\nTIGHT highest predicted mean:")
print("candidate =", tight_candidates[mean_idx])
print("mean =", tight_mu[mean_idx])
print("std =", tight_sigma[mean_idx])


# ------------------------------------------------------------
# UCB
# ------------------------------------------------------------

print("\nTIGHT UCB:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = tight_mu + beta * tight_sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", tight_candidates[idx],
        "\n mean =", round(tight_mu[idx], 6),
        "\n std =", round(tight_sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )

Best observed: [0.442929 0.409333 0.63183  0.7398   0.129381]
Tight half-widths: [0.05 0.05 0.05 0.05 0.05]
Lower: [0.392929 0.359333 0.58183  0.6898   0.079381]
Upper: [0.492929 0.459333 0.68183  0.7898   0.179381]

TIGHT EI:
candidate = [0.44280424 0.36106128 0.59342476 0.68992277 0.11972641]
mean = -0.21116778423149662
std = 0.04842856141182731
EI = 0.012368033097799429

TIGHT highest predicted mean:
candidate = [0.44048304 0.38520819 0.59513144 0.6898989  0.12001382]
mean = -0.20942664002060984
std = 0.04350495402069052

TIGHT UCB:

beta=0.1 
 candidate = [0.4371267  0.38041056 0.59018641 0.691066   0.12584443] 
 mean = -0.209508 
 std = 0.044556 
 UCB = -0.205052 

beta=0.25 
 candidate = [0.44026355 0.37192991 0.59085757 0.6902412  0.13151323] 
 mean = -0.209874 
 std = 0.046203 
 UCB = -0.198323 

beta=0.5 
 candidate = [0.44250603 0.36885039 0.58465143 0.69045385 0.13364565] 
 mean = -0.21022 
 std = 0.046947 
 UCB = -0.186746 

beta=1.0 
 candidate = [0.44280424 0.36106128 0.5

In [3]:
# ============================================================
# FINAL FUNCTION 6 - WEEK 9 SELECTION
# ============================================================
#
# Week 8 missed by -3.88 predictive standard deviations,
# so aggressive exploration is deliberately avoided.
#
# A tighter ARD trust region was used around the best
# actually observed point.
#
# beta = 0.25 gives modest exploration while staying
# close to the locally supported region.

beta = 0.25

UCB = tight_mu + beta * tight_sigma
final_idx = np.argmax(UCB)

week9_candidate = tight_candidates[final_idx]

print("Week 9 Function 6 candidate:")
print(week9_candidate)

print("\nPredicted mean:")
print(tight_mu[final_idx])

print("\nPredicted std:")
print(tight_sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 6 candidate:
[0.44026355 0.37192991 0.59085757 0.6902412  0.13151323]

Predicted mean:
-0.20987366871108748

Predicted std:
0.046203376714395984

UCB:
-0.19832282453248848

Portal format:
0.440264-0.371930-0.590858-0.690241-0.131513
